# Checking a recording · Shravya
**Project 24 · Activity Recognition System · Review 2.** Run the cells top to bottom (Jupyter: kernel *har*; Colab: run the setup cell first, it fetches the data and models).

In [1]:
import os, sys, subprocess, pathlib, urllib.request, zipfile, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/activity-recognition"):
        subprocess.run(["git", "clone", "-q", "https://github.com/rajyyug1132/activity-recognition", "/content/activity-recognition"], check=True)
    os.chdir("/content/activity-recognition")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe", "scikit-learn==1.9.1"], check=True)
    os.makedirs("data/uci", exist_ok=True); os.makedirs("models", exist_ok=True)
    if not os.path.exists("data/uci/UCI HAR Dataset"):
        urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip", "data/uci/har.zip")
        zipfile.ZipFile("data/uci/har.zip").extractall("data/uci")
        zipfile.ZipFile("data/uci/UCI HAR Dataset.zip").extractall("data/uci")
    if not os.path.exists("models/pose_landmarker_full.task"):
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task", "models/pose_landmarker_full.task")
    from google.colab import drive
    drive.mount("/content/drive")
    for f in ["cnn_uci.keras", "video_rf.joblib"]:
        if os.path.exists(f"/content/drive/MyDrive/{f}"):
            subprocess.run(["cp", f"/content/drive/MyDrive/{f}", f"models/{f}"], check=True)
elif pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")                      # run from the repository root

sys.path.insert(0, "src")
for f in ["config/config.yaml", "app/demo/uci_test_person_12.csv", "data/uci/UCI HAR Dataset",
          "models/cnn_uci.keras", "models/video_rf.joblib", "models/pose_landmarker_full.task"]:
    print("found  " if os.path.exists(f) else "MISSING", f)

found   config/config.yaml
found   app/demo/uci_test_person_12.csv
found   data/uci/UCI HAR Dataset
found   models/cnn_uci.keras
found   models/video_rf.joblib
found   models/pose_landmarker_full.task


In [2]:
%matplotlib inline
import inspect, yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Code, Image, display
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def show(fn):
    """Print a function's source exactly as it is in src/."""
    display(Code(inspect.getsource(fn), language="python"))

cfg = yaml.safe_load(open("config/config.yaml"))
DEMO = "app/demo/uci_test_person_12.csv"
NAMES = {"WALKING": "Walking", "WALKING_UPSTAIRS": "Upstairs", "WALKING_DOWNSTAIRS": "Downstairs",
         "SITTING": "Sitting", "STANDING": "Standing", "LAYING": "Lying"}
COLOURS = {"WALKING": "#2bb3a3", "WALKING_DOWNSTAIRS": "#2b3a8f", "WALKING_UPSTAIRS": "#7fb8e6",
           "SITTING": "#e9853a", "STANDING": "#e8c547", "LAYING": "#7a2e6e"}

In [3]:
demo = pd.read_csv(DEMO)

## 3. Checking a recording before predicting · Shravya
**What to say**
- `validate_csv` runs before the model ever sees a file. The app shows its report: the estimated sampling rate, issues and warnings.
- It estimates the sampling rate from the median gap between timestamps, counts missing values, warns on gaps over 0.1 s and rejects gaps over 1 s. It also rejects missing columns, recordings shorter than one window, and values out of range, which usually means wrong units.
- Below, the real sample passes and three broken copies of it are caught.

In [4]:
from validation import validate_csv
show(validate_csv)

def validate_csv(df, cfg) -> dict:
    """Checks columns, sampling rate, missing readings. Returns {ok, issues, est_hz, warnings}; short gaps are warnings only."""
    issues, warnings, est_hz = [], [], None
    missing = [c for c in COLS if c not in df.columns]
    if missing:
        return {"ok": False, "est_hz": None, "issues": [f"missing columns: {missing}"]}

    n_nan = int(df[COLS].isna().sum().sum())
    if n_nan:
        issues.append(f"{n_nan} NaN values")
    d = df.dropna(subset=COLS)
    t = d["timestamp"].to_numpy(float)
    if len(t) < 2 or not np.all(np.diff(t) > 0):
        return {"ok": False, "est_hz": None, "issues": issues + ["too few rows or timestamps not increasing"]}

    gaps = np.diff(t)
    est_hz = float(1 / np.median(gaps))
    want = cfg["sampling_rate_hz"]
    if est_hz < (1 - HZ_TOL) * want:  # faster is fine: to_windows resamples to 50 Hz
        issues.append(f"sampling rate {est_hz:.1f} Hz, need >= {want} Hz")
    if np.percentile(gaps, 95) > 2 * np.median(gaps):  # ponytail: crude jitter test
        issues.append("uneven sampling (high gap spread)")
    n_gaps = int((gaps > SHORT_GAP).sum())
    if n_gaps:
        warnings.append(f"{n_gaps} time gaps > {SHORT_GAP}s (largest {gaps.max():.2f}s)")
    if gaps.max() > LONG_GAP:
        issues.append(f"long gap > {LONG_GAP}s: reject file")

    need = cfg["window_samples"] / want
    if t[-1] - t[0] < need:
        issues.append(f"only {t[-1] - t[0]:.2f}s of data, need >= {need:.2f}s")

    acc, gyro = d[["acc_x", "acc_y", "acc_z"]].to_numpy(float), d[["gyro_x", "gyro_y", "gyro_z"]].to_numpy(float)
    if np.abs(acc).max() > MAX_ACC_G:
        issues.append(f"|acc| up to {np.abs(acc).max():.1f} > {MAX_ACC_G} g (units?)")
    if np.abs(gyro).max() > MAX_GYRO:
        issues.append(f"|gyro| up to {np.abs(gyro).max():.1f} > {MAX_GYRO} rad/s (deg/s?)")
    if abs(np.median(np.linalg.norm(acc, axis=1)) - 9.8) < 1.0:
        issues.append("median |acc| ~9.8: looks like m/s^2, divide by 9.80665")

    return {"ok": not issues, "est_hz": est_hz, "issues": issues, "warnings": warnings}

In [5]:
def check(name, df):
    r = validate_csv(df, cfg)
    rate = f"{r['est_hz']:.0f} Hz" if r.get("est_hz") else "n/a"
    print(f"{name:40s} passes: {str(r['ok']):5s}  rate: {rate}")
    for m in r["issues"]:
        print("     issue:  ", m)
    for m in r.get("warnings", []):
        print("     warning:", m)

check("sample recording", demo)
check("gyro_z column missing", demo.drop(columns="gyro_z"))
check("2 seconds of readings lost", demo[(demo.timestamp < 30) | (demo.timestamp > 32)])
check("acceleration in m/s² instead of g", demo.assign(acc_x=demo.acc_x * 9.81, acc_y=demo.acc_y * 9.81, acc_z=demo.acc_z * 9.81))

sample recording                         passes: True   rate: 50 Hz
gyro_z column missing                    passes: False  rate: n/a
     issue:   missing columns: ['gyro_z']
2 seconds of readings lost               passes: False  rate: 50 Hz
     issue:   long gap > 1.0s: reject file
acceleration in m/s² instead of g        passes: False  rate: 50 Hz
     issue:   |acc| up to 19.9 > 8.0 g (units?)
     issue:   median |acc| ~9.8: looks like m/s^2, divide by 9.80665
